In [24]:
import copy
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import optuna
import optuna
import torch
import torch.nn as nn
from matplotlib.patches import Circle, FancyArrowPatch, FancyBboxPatch
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import (
    GroupShuffleSplit,
    StratifiedGroupKFold,
)
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset
from xgboost import XGBClassifier
import plotly.express as px
from plotly.subplots import make_subplots

RANDOM_STATE = 151126

np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print("Device:", DEVICE)


Device: mps


## Part XIX — Hometask: VGG3D embeddings and cell-type classification

Use the alternative VGG3D representation of the same microscopy experiment:

```text
week6/large_object_vgg3d_embeddings_processed.csv
```

The target is `cell_type` with classes `m0`, `m1`, and `m2`. Do not use `sample_group` as a predictive feature; use it only to create leakage-safe partitions.

## Instructions

1. Inspect the table and complete a short preparation checklist: shape, data types, class balance, missingness, duplicate rows, feature ranges, and the meaning of `sample_group`.
2. Perform exploratory analysis with feature distributions and a PCA plus t-SNE or UMAP plot coloured by `cell_type`. Treat these projections as exploratory, not as test-set evaluation.
3. Create group-aware train, validation, and test partitions. Fit imputation and scaling on the training partition only, and verify that no `sample_group` occurs in more than one partition.
4. Train a multinomial logistic-regression baseline and a PyTorch MLP. Report learning curves, balanced accuracy, macro-F1, and a confusion matrix.
5. Run controlled MLP experiments: at least three learning rates, two network capacities, and two regularisation settings involving weight decay, dropout, or early stopping. Keep the split and random seed fixed.
6. Select the final model using validation data, evaluate it once on the untouched test set, and explain the errors biologically and statistically.

## Questions for the report

1. What does the VGG3D embedding table contain, and what data-preparation problems did you find?
2. Does the embedding space show separation between `m0`, `m1`, and `m2`? Is the separation also visible across sample groups?
3. How much does the MLP improve over logistic regression, and what evidence supports the comparison?
4. Which learning rate, capacity, and regularisation setting worked best? Did the MLP overfit?
5. Which cell types are confused most often, and do the errors concentrate in particular sample groups?
6. What biological or technical information is missing that could help explain the remaining errors?

## Deliverables

- completed notebook;
- preparation and leakage checks;
- PCA and t-SNE/UMAP visualisations;
- model comparison and learning-curve figures;
- final confusion matrix and six-question report.


In [6]:
data = pd.read_csv("large_object_vgg3d_embeddings_processed.csv")
data

,cell_type,sample_group,vgg_emb_000,vgg_emb_001,vgg_emb_002,vgg_emb_003,vgg_emb_004,vgg_emb_005,vgg_emb_006,vgg_emb_007,...,vgg_emb_118,vgg_emb_119,vgg_emb_120,vgg_emb_121,vgg_emb_122,vgg_emb_123,vgg_emb_124,vgg_emb_125,vgg_emb_126,vgg_emb_127
0,m0,sample_group_079,0.039772,0.0,1.179048,0.0,0.0,0.0,1.221995,0.0,...,0.0,0.0,0.0,0.797191,1.709197,0.0,0.777199,0.0,1.924863,0.0
1,m0,sample_group_079,0.000000,0.0,4.433527,0.0,0.0,0.0,5.810353,0.0,...,0.0,0.0,0.0,3.935624,6.289729,0.0,2.987612,0.0,8.424339,0.0
2,m0,sample_group_079,0.000000,0.0,4.108049,0.0,0.0,0.0,5.343654,0.0,...,0.0,0.0,0.0,3.621442,5.779456,0.0,2.740911,0.0,7.669261,0.0
3,m0,sample_group_079,0.000000,0.0,4.042331,0.0,0.0,0.0,5.283560,0.0,...,0.0,0.0,0.0,3.587561,5.783239,0.0,2.660382,0.0,7.480461,0.0
4,m0,sample_group_079,0.000000,0.0,2.474535,0.0,0.0,0.0,3.063431,0.0,...,0.0,0.0,0.0,2.021839,3.586300,0.0,1.637182,0.0,4.486126,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
376,m2,sample_group_056,0.000000,0.0,2.634610,0.0,0.0,0.0,3.325004,0.0,...,0.0,0.0,0.0,2.133105,3.855358,0.0,1.736552,0.0,4.645347,0.0
377,m2,sample_group_012,0.000000,0.0,1.678640,0.0,0.0,0.0,1.937712,0.0,...,0.0,0.0,0.0,1.231254,2.411791,0.0,1.117645,0.0,2.879727,0.0
378,m2,sample_group_012,0.000000,0.0,3.090644,0.0,0.0,0.0,3.945675,0.0,...,0.0,0.0,0.0,2.665728,4.402645,0.0,2.081455,0.0,5.807134,0.0
379,m2,sample_group_012,0.000000,0.0,2.557516,0.0,0.0,0.0,3.187129,0.0,...,0.0,0.0,0.0,2.117242,3.617540,0.0,1.689103,0.0,4.571338,0.0


> Inspect the table and complete a short preparation checklist: shape, data types, class balance, missingness, duplicate rows, feature ranges, and the meaning of sample_group.

sample_group - images from different experiments/batches

In [7]:
data.isna().sum()

cell_type       0
sample_group    0
vgg_emb_000     0
vgg_emb_001     0
vgg_emb_002     0
               ..
vgg_emb_123     0
vgg_emb_124     0
vgg_emb_125     0
vgg_emb_126     0
vgg_emb_127     0
Length: 130, dtype: int64

no na, nice

In [8]:
fig = px.histogram(data, x="cell_type", color="cell_type", title="Class distribution")
fig.show()

In [9]:
group_counts = pd.crosstab(data["sample_group"], data["cell_type"]).sort_values(by="sample_group")
print("n groups:", data["sample_group"].nunique())
print("classes per group:\n", (group_counts > 0).sum(axis=1).value_counts())

fig = px.histogram(data, x="sample_group", color="cell_type", title="Classes per sample group")
fig.show()

n groups: 85
classes per group:
 1    85
Name: count, dtype: int64


one sample group only has one dedicated class but several entries, also might indicated the cell_type relation

In [10]:
print("duplicated rows:", data.duplicated().sum())
print("duplicated embeddings:", data.drop(columns=["cell_type", "sample_group"]).duplicated().sum())

duplicated rows: 0
duplicated embeddings: 0


Perform exploratory analysis with feature distributions and a PCA plus t-SNE or UMAP plot coloured by cell_type. Treat these projections as exploratory, not as test-set evaluation.

In [11]:
X_emb = data.drop(columns=["cell_type", "sample_group"])

In [12]:
pca = PCA(random_state=RANDOM_STATE)
pca_xy = pca.fit_transform(X_emb)
var = pca.explained_variance_ratio_ * 100
print(f"PC1: {var[0]:.2f}%, PC2: {var[1]:.2f}%, PC1+PC2: {var[:2].sum():.2f}%")

scatter = px.scatter(x=pca_xy[:, 0], y=pca_xy[:, 1], color=data["cell_type"])
elbow = px.line(x=np.arange(1, len(var) + 1), y=var, markers=True)

fig = make_subplots(rows=1, cols=2, subplot_titles=["PCA", "PCA elbow"])
fig.add_traces(scatter.data, rows=1, cols=1)
fig.add_traces(elbow.data, rows=1, cols=2)
fig.update_xaxes(title_text=f"PC1 ({var[0]:.1f}%)", row=1, col=1)
fig.update_yaxes(title_text=f"PC2 ({var[1]:.1f}%)", row=1, col=1)
fig.update_xaxes(title_text="component", row=1, col=2)
fig.update_yaxes(title_text="explained variance, %", row=1, col=2)
fig.update_layout(title="PCA by cell_type", legend_title_text="cell_type")
fig.show()

PC1: 99.93%, PC2: 0.04%, PC1+PC2: 99.97%


pca has all of the variance in one component

In [13]:
perplexities = [5, 30, 50]
proj = []
for p in perplexities:
    tsne_xy = TSNE(n_components=2, perplexity=p, random_state=RANDOM_STATE).fit_transform(X_emb)
    proj.append(pd.DataFrame({"x": tsne_xy[:, 0], "y": tsne_xy[:, 1], "cell_type": data["cell_type"], "perplexity": p}))
proj = pd.concat(proj)

fig = px.scatter(proj, x="x", y="y", color="cell_type", facet_col="perplexity", title="t-SNE by cell_type")
fig.update_xaxes(matches=None)
fig.update_yaxes(matches=None)
fig.show()

in both tsne(p=30) and pca we see a clear separateion  m0 and m1 classes, and m2 is kinda over both of them

In [14]:
X = X_emb.copy()
y = data["cell_type"].copy()
groups = data["sample_group"].copy()

outer_splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
train_idx, test_idx = next(outer_splitter.split(X, y, groups=groups))

X_train_full = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()
y_train_full = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()
groups_train_full = groups.iloc[train_idx].copy()
groups_test = groups.iloc[test_idx].copy()

inner_splitter = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=RANDOM_STATE + 1)
train_idx2, val_idx = next(inner_splitter.split(X_train_full, y_train_full, groups=groups_train_full))

X_train = X_train_full.iloc[train_idx2].copy()
X_val = X_train_full.iloc[val_idx].copy()
y_train = y_train_full.iloc[train_idx2].copy()
y_val = y_train_full.iloc[val_idx].copy()
groups_train = groups_train_full.iloc[train_idx2].copy()
groups_val = groups_train_full.iloc[val_idx].copy()

print("objects train/val/test:", len(X_train), len(X_val), len(X_test))
print("groups train/val/test:", groups_train.nunique(), groups_val.nunique(), groups_test.nunique())
print("shared train/val:", len(set(groups_train) & set(groups_val)))
print("shared train/test:", len(set(groups_train) & set(groups_test)))
print("shared val/test:", len(set(groups_val) & set(groups_test)))

print(pd.DataFrame({
    "all": y.value_counts(normalize=True),
    "train": y_train.value_counts(normalize=True),
    "val": y_val.value_counts(normalize=True),
    "test": y_test.value_counts(normalize=True),
}).sort_index())

assert len(set(groups_train) & set(groups_val)) == 0
assert len(set(groups_train) & set(groups_test)) == 0
assert len(set(groups_val) & set(groups_test)) == 0

objects train/val/test: 228 76 77
groups train/val/test: 52 16 17
shared train/val: 0
shared train/test: 0
shared val/test: 0
                all     train       val      test
cell_type                                        
m0         0.346457  0.346491  0.355263  0.337662
m1         0.272966  0.271930  0.276316  0.272727
m2         0.380577  0.381579  0.368421  0.389610


In [22]:
def show_metrics(name, y_true, y_pred):
    values = {
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
    }
    print(name)
    print(f"  balanced accuracy: {values['balanced_accuracy']:.3f}")
    print(f"  macro-F1:          {values['macro_f1']:.3f}")
    return values

In [23]:
from sklearn.pipeline import Pipeline

logreg = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
])
logreg.fit(X_train, y_train)

logreg_val_metrics = show_metrics("LogReg val", y_val, logreg.predict(X_val))

LogReg val
  balanced accuracy: 0.727
  macro-F1:          0.712


In [28]:
import lightning as L

class MLPModule(L.LightningModule):
    def __init__(self, model, lr=1e-3, weight_decay=0.0):
        super().__init__()
        self.model = model
        self.lr = lr
        self.weight_decay = weight_decay
        self.criterion = nn.CrossEntropyLoss()
        self.history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        X_batch, y_batch = batch
        logits = self(X_batch)
        loss = self.criterion(logits, y_batch)
        self.log("train_loss", loss, on_step=False, on_epoch=True)
        self.log("train_acc", (logits.argmax(dim=1) == y_batch).float().mean(), on_step=False, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        X_batch, y_batch = batch
        logits = self(X_batch)
        self.log("val_loss", self.criterion(logits, y_batch), on_step=False, on_epoch=True)
        self.log("val_acc", (logits.argmax(dim=1) == y_batch).float().mean(), on_step=False, on_epoch=True)

    def on_train_epoch_end(self):
        for key in self.history:
            self.history[key].append(self.trainer.callback_metrics[key].item())

    def predict_step(self, batch, batch_idx):
        return self(batch[0]).argmax(dim=1)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.lr, weight_decay=self.weight_decay)

In [29]:
class MLP(nn.Module):
    def __init__(self, input_dim, n_classes):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, n_classes)
        )

    def forward(self, x):
        return self.network(x)

In [30]:
class_names = sorted(y.unique())

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
X_val_t = torch.tensor(X_val_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train.map(class_names.index).to_numpy(), dtype=torch.long)
y_val_t = torch.tensor(y_val.map(class_names.index).to_numpy(), dtype=torch.long)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=32, shuffle=False)

In [31]:
mlp = MLPModule(MLP(X_train_t.shape[1], len(class_names)), lr=1e-3)
trainer = L.Trainer(max_epochs=100, accelerator="auto", logger=False, enable_checkpointing=False,
                    enable_progress_bar=False, enable_model_summary=False, num_sanity_val_steps=0)
trainer.fit(mlp, train_loader, val_loader)

mlp_val_pred = [class_names[i] for i in torch.cat(trainer.predict(mlp, val_loader)).cpu().numpy()]

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/Users/neprog/Desktop/kse/ml_bio/ml_bio_hw/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/Users/neprog/Desktop/kse/ml_bio/ml_bio_hw/.venv/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
/Users/neprog/Desktop/kse/ml_bio/ml_bio_hw/.venv/lib/python3.11/site-packages/lightning/pytorch/trainer/connector

In [32]:
mlp_val_metrics = show_metrics("MLP val", y_val, mlp_val_pred)

fig = px.line(pd.DataFrame(mlp.history), markers=True, title="MLP: learning curves",
              labels={"index": "epoch", "value": "value", "variable": "curve"})
fig.show()

cm = confusion_matrix(y_val, mlp_val_pred, labels=class_names)
fig = px.imshow(cm, x=class_names, y=class_names, text_auto=True, color_continuous_scale="Blues",
                labels={"x": "predicted", "y": "true", "color": "count"},
                title="MLP val: confusion matrix")

MLP val
  balanced accuracy: 0.691
  macro-F1:          0.688


iu, performed worse than logreg, was training up until 8th epoch, then val loss started increasing while accurasy stayed kinda the same thoughout all of the epochs. That means that model became more and more confident in its wrong predictions

In [33]:
class FlexibleMLP(nn.Module):
    def __init__(self, input_dim, n_classes, hidden_dims=(32, 16), dropout=0.0):
        super().__init__()

        layers = []
        current_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(current_dim, hidden_dim))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            current_dim = hidden_dim

        layers.append(nn.Linear(current_dim, n_classes))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

In [53]:
from lightning.pytorch.callbacks import ModelCheckpoint

experiments = [
    {"lr": 1e-4, "hidden_dims": (32, 16),  "dropout": 0.0, "weight_decay": 0.0},
    {"lr": 1e-3, "hidden_dims": (32, 16),  "dropout": 0.0, "weight_decay": 0.0},
    {"lr": 1e-2, "hidden_dims": (32, 16),  "dropout": 0.0, "weight_decay": 0.0},
    {"lr": 1e-3, "hidden_dims": (128, 64), "dropout": 0.0, "weight_decay": 0.0},
    {"lr": 1e-3, "hidden_dims": (32, 16),  "dropout": 0.3, "weight_decay": 0.0},
    {"lr": 1e-3, "hidden_dims": (32, 16),  "dropout": 0.0, "weight_decay": 1e-3},
]

results = []
for params in experiments:
    L.seed_everything(RANDOM_STATE, verbose=False)
    model = MLPModule(FlexibleMLP(X_train_t.shape[1], len(class_names), params["hidden_dims"], params["dropout"]),
                      lr=params["lr"], weight_decay=params["weight_decay"])
    checkpoint = ModelCheckpoint(monitor="val_loss", mode="min", dirpath="checkpoints", filename="best",
                                 save_top_k=1, enable_version_counter=False)
    trainer = L.Trainer(max_epochs=100, accelerator="auto", logger=False, callbacks=[checkpoint],
                        enable_progress_bar=False, enable_model_summary=False, num_sanity_val_steps=0)
    trainer.fit(model, train_loader, val_loader)

    pred = [class_names[i] for i in torch.cat(trainer.predict(model, val_loader, ckpt_path="best")).cpu().numpy()]
    results.append({**params,
                    "best_epoch": int(np.argmin(model.history["val_loss"])) + 1,
                    "val_bal_acc": balanced_accuracy_score(y_val, pred),
                    "val_macro_f1": f1_score(y_val, pred, average="macro"),
                    "history": model.history})


GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


`Trainer.fit` stopped: `max_epochs=100` reached.
Restoring states from the checkpoint path at /Users/neprog/Desktop/kse/ml_bio/ml_bio_hw/lab6/checkpoints/best.ckpt
Loaded model weights from the checkpoint at /Users/neprog/Desktop/kse/ml_bio/ml_bio_hw/lab6/checkpoints/best.ckpt
/Users/neprog/Desktop/kse/ml_bio/ml_bio_hw/.venv/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/Users/neprog/Desktop/kse/ml_bio/ml_bio_hw/.venv/lib/python3.11/site-pa

In [54]:
results = pd.DataFrame(results)
results.drop(columns="history").sort_values("val_bal_acc", ascending=False)

,lr,hidden_dims,dropout,weight_decay,best_epoch,val_bal_acc,val_macro_f1
4,0.0010,"(32, 16)",0.3,0.000,24,0.707231,0.675815
1,0.0010,"(32, 16)",0.0,0.000,10,0.639771,0.604181
0,0.0001,"(32, 16)",0.0,0.000,77,0.627866,0.588183
5,0.0010,"(32, 16)",0.0,0.001,9,0.615520,0.577310
2,0.0100,"(32, 16)",0.0,0.000,1,0.604056,0.550255
3,0.0010,"(128, 64)",0.0,0.000,2,0.580688,0.498858


In [55]:
curves = []
for i, row in results.iterrows():
    h = pd.DataFrame(row["history"])
    h["epoch"] = range(len(h))
    h["run"] = f"lr={row['lr']}, {row['hidden_dims']}, do={row['dropout']}, wd={row['weight_decay']}"
    curves.append(h)
curves = pd.concat(curves).melt(id_vars=["epoch", "run"], var_name="curve", value_name="value")

fig = px.line(curves, x="epoch", y="value", color="curve", facet_col="run", facet_col_wrap=3,
              log_y=True, height=1000, title="MLP experiments: learning curves")
fig.update_yaxes(tickvals=[0.3, 0.5, 1, 2, 5, 10, 20, 50], ticktext=["0.3", "0.5", "1", "2", "5", "10", "20", "50"])
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=", 1)[1]))
fig.show()

the best one with the 128/64 dim, which is expected as more pattersn might be learned with change of overfitting(the best model was on the 1 epoch)


ps at this point i realized that model(weight) were taken from the last epoch, not from the best one, so i rerun everything with taking the best epoch checkpoint

after the rerun the best model was lr 0.001, dim 32/16, 0.3 dropout and 0 decay at epoch 22. this model might've been just lucky, from the curves we can we a small dip at this epoch relative to the neighbours 18/28, still the best one. 128th overfitted right away and was performing poorly on val

ps2 at this point i realized mb that val loss might not be the target to focus on, but val_acc should be the one. It really depends what is more important - the overall "rightness" of the model or the result(even if it means it more confidently predict wrong values), so the experiment chosing the epoch based on the acc is the next one

In [57]:
acc_results = []
for params in experiments:
    L.seed_everything(RANDOM_STATE, verbose=False)
    model = MLPModule(FlexibleMLP(X_train_t.shape[1], len(class_names), params["hidden_dims"], params["dropout"]),
                      lr=params["lr"], weight_decay=params["weight_decay"])
    checkpoint = ModelCheckpoint(monitor="val_acc", mode="max", dirpath="checkpoints", filename="best",
                                 save_top_k=1, enable_version_counter=False)
    trainer = L.Trainer(max_epochs=100, accelerator="auto", logger=False, callbacks=[checkpoint],
                        enable_progress_bar=False, enable_model_summary=False, num_sanity_val_steps=0)
    trainer.fit(model, train_loader, val_loader)

    pred = [class_names[i] for i in torch.cat(trainer.predict(model, val_loader, ckpt_path="best")).cpu().numpy()]
    acc_results.append({**params,
                        "best_epoch": int(np.argmax(model.history["val_acc"])) + 1,
                        "val_bal_acc": balanced_accuracy_score(y_val, pred),
                        "val_macro_f1": f1_score(y_val, pred, average="macro"),
                        "history": model.history})



GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/Users/neprog/Desktop/kse/ml_bio/ml_bio_hw/.venv/lib/python3.11/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /Users/neprog/Desktop/kse/ml_bio/ml_bio_hw/lab6/checkpoints exists and is not empty.
/Users/neprog/Desktop/kse/ml_bio/ml_bio_hw/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/Users/neprog/Desktop/kse/ml_bio/ml_bio_hw/.venv/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. C

In [58]:
acc_results = pd.DataFrame(acc_results)
acc_results.drop(columns="history").sort_values("val_bal_acc", ascending=False)

,lr,hidden_dims,dropout,weight_decay,best_epoch,val_bal_acc,val_macro_f1
3,0.0010,"(128, 64)",0.0,0.000,93,0.754850,0.736209
4,0.0010,"(32, 16)",0.3,0.000,29,0.742945,0.721344
2,0.0100,"(32, 16)",0.0,0.000,50,0.719136,0.724279
5,0.0010,"(32, 16)",0.0,0.001,63,0.719136,0.712449
1,0.0010,"(32, 16)",0.0,0.000,99,0.714727,0.714148
0,0.0001,"(32, 16)",0.0,0.000,88,0.639771,0.604630


here the best one is the 128/64 at 93 epoch. even tho it has a high val_loss that was flagged at the prev stage - its accuracy is the best so far. In our case it means that it does not overfit on data itself, but rather it is more confident in its right and wrong guesses

In [62]:
new_experiments = [
    {"lr": 1e-3, "hidden_dims": (128, 64), "dropout": 0.0, "weight_decay": 0.0},
    {"lr": 1e-3, "hidden_dims": (256, 128), "dropout": 0.0, "weight_decay": 0.0},
    {"lr": 1e-4, "hidden_dims": (128, 64),  "dropout": 0.0, "weight_decay": 0.0},
    {"lr": 1e-2, "hidden_dims": (128, 64),  "dropout": 0.0, "weight_decay": 0.0},
    {"lr": 1e-3, "hidden_dims": (128, 64),  "dropout": 0.3, "weight_decay": 0.0},
    {"lr": 1e-3, "hidden_dims": (128, 64),  "dropout": 0.0, "weight_decay": 1e-3},
    {"lr": 1e-3, "hidden_dims": (128, 64),  "dropout": 0.3, "weight_decay": 1e-3},
]

new_results = []
for params in new_experiments:
    L.seed_everything(RANDOM_STATE, verbose=False)
    model = MLPModule(FlexibleMLP(X_train_t.shape[1], len(class_names), params["hidden_dims"], params["dropout"]),
                      lr=params["lr"], weight_decay=params["weight_decay"])
    checkpoint = ModelCheckpoint(monitor="val_acc", mode="max", dirpath="checkpoints", filename="best",
                                 save_top_k=1, enable_version_counter=False)
    trainer = L.Trainer(max_epochs=100, accelerator="auto", logger=False, callbacks=[checkpoint],
                        enable_progress_bar=False, enable_model_summary=False, num_sanity_val_steps=0)
    trainer.fit(model, train_loader, val_loader)

    pred = [class_names[i] for i in torch.cat(trainer.predict(model, val_loader, ckpt_path="best")).cpu().numpy()]
    new_results.append({**params,
                        "best_epoch": int(np.argmax(model.history["val_acc"])) + 1,
                        "val_bal_acc": balanced_accuracy_score(y_val, pred),
                        "val_macro_f1": f1_score(y_val, pred, average="macro"),
                        "history": model.history})

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/Users/neprog/Desktop/kse/ml_bio/ml_bio_hw/.venv/lib/python3.11/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /Users/neprog/Desktop/kse/ml_bio/ml_bio_hw/lab6/checkpoints exists and is not empty.
/Users/neprog/Desktop/kse/ml_bio/ml_bio_hw/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/Users/neprog/Desktop/kse/ml_bio/ml_bio_hw/.venv/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. C

In [63]:
new_results = pd.DataFrame(new_results)
new_results.drop(columns="history").sort_values("val_bal_acc", ascending=False)

,lr,hidden_dims,dropout,weight_decay,best_epoch,val_bal_acc,val_macro_f1
0,0.0010,"(128, 64)",0.0,0.000,93,0.754850,0.736209
4,0.0010,"(128, 64)",0.3,0.000,22,0.753968,0.738857
1,0.0010,"(256, 128)",0.0,0.000,29,0.750882,0.737949
2,0.0001,"(128, 64)",0.0,0.000,55,0.742945,0.720749
3,0.0100,"(128, 64)",0.0,0.000,6,0.742945,0.721344
6,0.0010,"(128, 64)",0.3,0.001,21,0.742945,0.721344
5,0.0010,"(128, 64)",0.0,0.001,68,0.738536,0.725673


the best model is still 128/64(even tho dropout one is kinda close), so we will stick with it

In [64]:
L.seed_everything(RANDOM_STATE, verbose=False)
final_model = MLPModule(FlexibleMLP(X_train_t.shape[1], len(class_names), (128, 64), 0.0), lr=1e-3, weight_decay=0.0)
final_checkpoint = ModelCheckpoint(monitor="val_acc", mode="max", dirpath="checkpoints", filename="final",
                                   save_top_k=1, enable_version_counter=False)
final_trainer = L.Trainer(max_epochs=100, accelerator="auto", logger=False, callbacks=[final_checkpoint],
                          enable_progress_bar=False, enable_model_summary=False, num_sanity_val_steps=0)
final_trainer.fit(final_model, train_loader, val_loader)
print("best epoch:", int(np.argmax(final_model.history["val_acc"])) + 1)

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/Users/neprog/Desktop/kse/ml_bio/ml_bio_hw/.venv/lib/python3.11/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /Users/neprog/Desktop/kse/ml_bio/ml_bio_hw/lab6/checkpoints exists and is not empty.
/Users/neprog/Desktop/kse/ml_bio/ml_bio_hw/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/Users/neprog/Desktop/kse/ml_bio/ml_bio_hw/.venv/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. C

best epoch: 93


In [65]:
final_model = MLPModule(FlexibleMLP(X_train_t.shape[1], len(class_names), (128, 64), 0.0))
final_model.load_state_dict(torch.load("checkpoints/final.ckpt", weights_only=False)["state_dict"])
final_model.eval()

X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32)
with torch.no_grad():
    test_pred = [class_names[i] for i in final_model(X_test_t).argmax(dim=1).numpy()]

test_metrics = show_metrics("Final MLP test", y_test, test_pred)

cm = confusion_matrix(y_test, test_pred, labels=class_names)
fig = px.imshow(cm, x=class_names, y=class_names, text_auto=True, color_continuous_scale="Blues",
                labels={"x": "predicted", "y": "true", "color": "count"},
                title="Final MLP test: confusion matrix")
fig.show()

Final MLP test
  balanced accuracy: 0.673
  macro-F1:          0.671


score is worse then on val, there could be a couple of reasons for that: the overall training set is quite small and test is somewhat different from the training(even tho i've done stratified grouped)
we can see that a lot of mistakes are m0 true -> m2 predicted and we saw that on both PCA and T-SNE there is an overlap between those two classes

What does the VGG3D embedding table contain, and what data-preparation problems did you find?  
i didnt find any, tbh, no na/duplications

Does the embedding space show separation between m0, m1, and m2? Is the separation also visible across sample groups? 
m1 is separate and m0/m2 share some variability(and that we can see on the final confusion matrix)

How much does the MLP improve over logistic regression, and what evidence supports the comparison?   
complicated question, we had one MLP model that performed better than logreg on the val dataset  
MLP performed worse on the test dataset, as well as logreg(can see down below the result)  

so MLP is better, by around 0.07

In [66]:
logreg_test_pred = logreg.predict(X_test)
logreg_test_metrics = show_metrics("LogReg test", y_test, logreg_test_pred)

cm = confusion_matrix(y_test, logreg_test_pred, labels=class_names)
fig = px.imshow(cm, x=class_names, y=class_names, text_auto=True, color_continuous_scale="Blues",
                labels={"x": "predicted", "y": "true", "color": "count"},
                title="LogReg test: confusion matrix")
fig.show()

LogReg test
  balanced accuracy: 0.603
  macro-F1:          0.601


Which learning rate, capacity, and regularisation setting worked best? Did the MLP overfit?  
128/64, dropout 0, w_d 0, lr 0.001  
based on the val_loss - overfitter from the 2nd epoch, but the best val_accuracy it gave on 93, even tho it was highly overfitted. it became more confident, both in right and wrong answers

Which cell types are confused most often, and do the errors concentrate in particular sample groups?  
m0->m2, m2->m0, as they share a lot variability 
by sample groups - its kinda complicated as groups share different number of samples and they represent different classes(like 021 that is m2 with 22 samples inside), so by proportion and by abs value the answer might differ

In [70]:
errors = (y_test != test_pred).groupby(groups_test).agg(["sum", "size", "mean"])
errors.columns = ["errors", "n_samples", "error_rate"]
errors["cell_type"] = y_test.groupby(groups_test).first()
errors.sort_values("errors", ascending=False)

,errors,n_samples,error_rate,cell_type
sample_group,,,,
sample_group_021,7,22,0.318182,m2
sample_group_062,4,11,0.363636,m0
sample_group_026,3,7,0.428571,m0
sample_group_054,2,2,1.000000,m2
sample_group_019,2,4,0.500000,m1
sample_group_006,2,2,1.000000,m0
sample_group_045,2,6,0.333333,m0
sample_group_007,1,1,1.000000,m1
sample_group_012,1,4,0.250000,m2


What biological or technical information is missing that could help explain the remaining errors?  
what are those classes, are they really different, what potentially makes them in the similar emb space